# Module 00: Benchmark Datasets for the Course

## Learning Objectives

By completing this notebook, you will:
1. Load and profile five benchmark datasets used throughout the course
2. Understand the key properties of each dataset: dimensionality, class balance, feature types, and the selection challenge each presents
3. Establish baseline model performance (Random Forest and LightGBM) on each dataset
4. Have reusable data loading functions ready to import in later modules

## Prerequisites
- Notebook 01 completed (introduced the Madelon dataset and core concepts)
- scikit-learn, lightgbm, numpy, pandas installed

## Estimated Time: 15 minutes

## The Benchmark Suite

We use five datasets that together cover the range of feature selection challenges:

| Dataset | n | p | Task | Challenge |
|---|---|---|---|---|
| Madelon | 2,600 | 500 | Binary clf | Known informative features (20/500) |
| MNIST (784-dim) | 70,000 | 784 | Multiclass clf | Dense image features, high n |
| Breast Cancer Wisconsin | 569 | 30 | Binary clf | Small n, correlated features |
| Communities & Crime | 1,994 | 122 | Regression | Missing values, mixed relevance |
| Connectionist Bench (Sonar) | 208 | 60 | Binary clf | Small n, high p/n |

Each dataset highlights different aspects of the feature selection problem and will be revisited in later modules.

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import (
    fetch_openml,
    load_breast_cancer,
    make_classification,
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# Try importing LightGBM — used for baselines, falls back to RF if unavailable
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
    print('LightGBM available.')
except ImportError:
    LGBM_AVAILABLE = False
    print('LightGBM not available — baselines will use Random Forest only.')

print('Imports complete.')

## Section 1: Data Loading Functions

We define a `DatasetLoader` class with a consistent interface for all five datasets. Each loader returns `(X, y, task_type, metadata)`. The same functions are imported by later modules — this is the reusable data layer for the entire course.

In [ ]:
class BenchmarkDataset:
    """Container for a benchmark dataset with metadata.
    
    Attributes
    ----------
    name : str
    X : ndarray of shape (n_samples, n_features) — cleaned, no missing values
    y : ndarray of shape (n_samples,)
    task : str — 'classification' or 'regression'
    feature_names : list of str
    n_classes : int or None (None for regression)
    n_informative_true : int or None — known ground truth if available
    description : str
    """
    def __init__(self, name, X, y, task, feature_names=None,
                 n_classes=None, n_informative_true=None, description=''):
        self.name = name
        self.X = np.array(X, dtype=np.float64)
        self.y = np.array(y)
        self.task = task
        self.feature_names = feature_names or [f'f{i}' for i in range(X.shape[1])]
        self.n_classes = n_classes
        self.n_informative_true = n_informative_true
        self.description = description

    @property
    def n(self):
        return self.X.shape[0]

    @property
    def p(self):
        return self.X.shape[1]

    @property
    def p_over_n(self):
        return self.p / self.n

    def class_balance(self):
        """Return dict of class label -> proportion, or None for regression."""
        if self.task == 'regression':
            return None
        unique, counts = np.unique(self.y, return_counts=True)
        return {str(u): c / len(self.y) for u, c in zip(unique, counts)}

    def has_missing(self):
        return bool(np.isnan(self.X).any())

    def summary(self):
        """Print a compact profile of the dataset."""
        print(f'Dataset: {self.name}')
        print(f'  Task:    {self.task}')
        print(f'  n:       {self.n:,}')
        print(f'  p:       {self.p}')
        print(f'  p/n:     {self.p_over_n:.4f}')
        if self.n_informative_true:
            print(f'  Known informative: {self.n_informative_true} / {self.p}')
        if self.task == 'classification':
            bal = self.class_balance()
            print(f'  Classes: {self.n_classes} | Balance: '
                  + ', '.join(f'{k}: {v:.1%}' for k, v in bal.items()))
        else:
            print(f'  y range: [{self.y.min():.3f}, {self.y.max():.3f}]')
        print(f'  Missing values: {self.has_missing()}')
        print(f'  Description: {self.description}')

print('BenchmarkDataset class defined.')

In [ ]:
def load_madelon() -> BenchmarkDataset:
    """Load Madelon from OpenML. Falls back to synthetic equivalent.
    
    Madelon (NIPS 2003 challenge): 500 features, 20 truly informative,
    480 noise features, binary classification.
    """
    try:
        ds = fetch_openml(name='madelon', version=1, as_frame=False, parser='auto')
        X = ds.data.astype(np.float64)
        y = LabelEncoder().fit_transform(ds.target)
    except Exception:
        X, y = make_classification(
            n_samples=2600, n_features=500, n_informative=20,
            n_redundant=10, n_clusters_per_class=2, random_state=42
        )
    return BenchmarkDataset(
        name='Madelon',
        X=X, y=y,
        task='classification',
        n_classes=2,
        n_informative_true=20,
        description=(
            'NIPS 2003 Feature Selection Challenge benchmark. '
            'Known ground truth: 20 informative, 480 noise features. '
            'Standard benchmark for selection method evaluation.'
        )
    )


def load_breast_cancer_wisconsin() -> BenchmarkDataset:
    """Load the Breast Cancer Wisconsin dataset from sklearn.
    
    569 samples, 30 features (all numeric, no missing values),
    binary classification (malignant / benign). Features are correlated
    (mean, std, worst of 10 nucleus measurements).
    """
    data = load_breast_cancer()
    return BenchmarkDataset(
        name='Breast Cancer Wisconsin',
        X=data.data,
        y=data.target,
        task='classification',
        feature_names=list(data.feature_names),
        n_classes=2,
        description=(
            'UCI classic medical dataset. 30 correlated features derived '
            'from 10 nucleus measurements (mean, std, worst). Small n=569. '
            'Tests methods for correlated feature handling.'
        )
    )


def load_sonar() -> BenchmarkDataset:
    """Load the Connectionist Bench (Sonar) dataset from OpenML.
    
    208 samples, 60 features (sonar frequency band energies),
    binary classification (rock vs mine). High p/n ratio = 0.29.
    """
    try:
        ds = fetch_openml(name='sonar', version=1, as_frame=False, parser='auto')
        X = ds.data.astype(np.float64)
        y = LabelEncoder().fit_transform(ds.target)
    except Exception:
        # Fallback: generate data with same dimensions
        X, y = make_classification(
            n_samples=208, n_features=60, n_informative=25,
            n_redundant=10, random_state=42
        )
    return BenchmarkDataset(
        name='Sonar',
        X=X, y=y,
        task='classification',
        n_classes=2,
        description=(
            'Connectionist Bench. 60 sonar frequency features. '
            'Small n=208 creates high p/n=0.29 — wrapper overfitting risk. '
            'Tests small-sample high-dimension handling.'
        )
    )


def load_mnist_subset() -> BenchmarkDataset:
    """Load MNIST 28x28 images as 784-dimensional vectors (10,000-sample subset).
    
    Pixel features are dense and correlated. Many zero-variance pixels at edges.
    Multiclass (10 classes). Tests variance/filter methods at moderate p.
    """
    try:
        print('  Fetching MNIST from OpenML (~14MB first run)...')
        ds = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
        # Use 10,000-sample subset for speed
        idx = np.random.RandomState(42).choice(len(ds.data), size=10000, replace=False)
        X = ds.data[idx].astype(np.float64)
        y = ds.target[idx].astype(int)
    except Exception:
        print('  OpenML failed — generating surrogate 784-dim dataset...')
        X, y = make_classification(
            n_samples=10000, n_features=784, n_informative=200,
            n_redundant=100, n_classes=10, n_clusters_per_class=1,
            random_state=42
        )
    return BenchmarkDataset(
        name='MNIST (10k subset)',
        X=X, y=y,
        task='classification',
        n_classes=10,
        description=(
            'MNIST handwritten digits. 784 pixel features (28x28). '
            '10,000-sample subset. Many zero-variance edge pixels. '
            'Dense correlated features; tests variance filters and image-aware selection.'
        )
    )


def load_communities_crime() -> BenchmarkDataset:
    """Load Communities and Crime dataset from OpenML.
    
    1,994 communities, 122 socio-economic features, regression target
    (violent crime rate per 100k population). Has missing values (~25% of features
    have some missing). Tests imputation + selection pipelines.
    """
    try:
        ds = fetch_openml(
            name='communities', version=1, as_frame=True, parser='auto'
        )
        df = ds.data
        # Convert to numeric, coercing errors to NaN
        df = df.apply(pd.to_numeric, errors='coerce')
        # Drop columns with > 50% missing (uninformative for selection demo)
        df = df.dropna(axis=1, thresh=int(0.5 * len(df)))
        # Impute remaining missing values with column median
        imputer = SimpleImputer(strategy='median')
        X = imputer.fit_transform(df)
        # Target: violent crimes per 100k population (normalised)
        y_raw = pd.to_numeric(ds.target, errors='coerce').values
        valid = ~np.isnan(y_raw)
        X, y = X[valid], y_raw[valid]
        feature_names = list(df.columns)
    except Exception:
        # Fallback: regression dataset with similar dimensions
        from sklearn.datasets import make_regression
        X, y = make_regression(
            n_samples=1994, n_features=122, n_informative=40,
            noise=0.2, random_state=42
        )
        feature_names = [f'socio_{i:03d}' for i in range(X.shape[1])]
    return BenchmarkDataset(
        name='Communities & Crime',
        X=X, y=y,
        task='regression',
        feature_names=feature_names if 'feature_names' in dir() else None,
        description=(
            'UCI socio-economic dataset. 122 community features predicting '
            'violent crime rate. Has missing values (handled by median imputation). '
            'Regression task — tests filter methods beyond classification.'
        )
    )


print('Loader functions defined.')

In [ ]:
# Load all five datasets
# Each loader handles OpenML fallback gracefully
print('Loading all benchmark datasets...')
print('=' * 55)

datasets = {}

print('Loading Madelon...')
datasets['madelon'] = load_madelon()

print('Loading Breast Cancer Wisconsin...')
datasets['breast_cancer'] = load_breast_cancer_wisconsin()

print('Loading Sonar...')
datasets['sonar'] = load_sonar()

print('Loading MNIST subset...')
datasets['mnist'] = load_mnist_subset()

print('Loading Communities & Crime...')
datasets['communities'] = load_communities_crime()

print(f'\nAll {len(datasets)} datasets loaded successfully.')

## Section 2: Dataset Profiles

A systematic profile of each dataset. We look at:
- Sample count ($n$), feature count ($p$), and the $p/n$ ratio
- Task type and class balance
- Feature distribution characteristics (variance spread, zero-variance features)
- The specific feature selection challenge each dataset presents

In [ ]:
# Print a detailed summary profile for each dataset
print('=' * 65)
for key, ds in datasets.items():
    ds.summary()
    print()

In [ ]:
# Build a compact comparison DataFrame — useful for reference throughout the course
def profile_dataset(ds: BenchmarkDataset) -> dict:
    """Extract key profiling statistics from a BenchmarkDataset."""
    # Feature variance statistics
    variances = np.var(ds.X, axis=0)
    zero_var = int((variances < 1e-10).sum())
    low_var = int((variances < np.percentile(variances, 10)).sum())

    # Feature correlation (mean absolute pairwise correlation, sampled for large p)
    if ds.p <= 100:
        corr_matrix = np.corrcoef(ds.X.T)
        upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
        mean_abs_corr = float(np.abs(upper_tri).mean())
    else:
        # For large p, sample 100 features to estimate correlation
        idx = np.random.choice(ds.p, size=min(100, ds.p), replace=False)
        corr_sub = np.corrcoef(ds.X[:, idx].T)
        upper_tri = corr_sub[np.triu_indices_from(corr_sub, k=1)]
        mean_abs_corr = float(np.abs(upper_tri).mean())

    balance_str = '—'
    if ds.task == 'classification':
        bal = ds.class_balance()
        min_class = min(bal.values())
        balance_str = f'{min_class:.1%} minority'

    return {
        'n': ds.n,
        'p': ds.p,
        'p/n': f'{ds.p_over_n:.3f}',
        'task': ds.task[:3],  # clf / reg
        'classes': ds.n_classes if ds.task == 'classification' else '—',
        'balance': balance_str,
        'zero_var_features': zero_var,
        'mean_abs_corr': f'{mean_abs_corr:.3f}',
        'known_informative': ds.n_informative_true or '?',
    }


profiles = {name: profile_dataset(ds) for name, ds in datasets.items()}
profile_df = pd.DataFrame(profiles).T

print('Benchmark Dataset Profiles:')
print(profile_df.to_string())

In [ ]:
# Visualise key profile dimensions for quick comparison
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

dataset_names = list(datasets.keys())
short_names = ['Madelon', 'BreastCa.', 'Sonar', 'MNIST', 'Communities']
colours = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#E91E63']

# Panel 1: n vs p scatter (log-log)
ax1 = fig.add_subplot(gs[0, 0])
for name, short, colour in zip(dataset_names, short_names, colours):
    ds = datasets[name]
    ax1.scatter(ds.n, ds.p, s=150, color=colour, zorder=5, label=short)
    ax1.annotate(short, (ds.n, ds.p), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)
# Draw p/n reference lines
n_range = np.logspace(2, 5, 100)
for ratio, style, label in [(0.1, '--', 'p/n=0.1'), (1.0, ':', 'p/n=1')]:
    ax1.plot(n_range, ratio * n_range, linestyle=style, color='grey',
             linewidth=1, alpha=0.6, label=label)
ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel('n (samples)', fontsize=10)
ax1.set_ylabel('p (features)', fontsize=10)
ax1.set_title('n vs. p landscape', fontsize=11)
ax1.legend(fontsize=7, loc='upper left')

# Panel 2: p/n ratio bar chart
ax2 = fig.add_subplot(gs[0, 1])
p_over_n_vals = [datasets[name].p_over_n for name in dataset_names]
ax2.bar(range(len(dataset_names)), p_over_n_vals, color=colours, alpha=0.85)
ax2.axhline(0.1, color='black', linestyle='--', linewidth=1.2, label='p/n=0.1 threshold')
ax2.axhline(1.0, color='crimson', linestyle=':', linewidth=1.2, label='p/n=1 (p>n regime)')
ax2.set_xticks(range(len(dataset_names)))
ax2.set_xticklabels(short_names, fontsize=9, rotation=20)
ax2.set_ylabel('p/n ratio', fontsize=10)
ax2.set_title('Dimensionality Risk (p/n)', fontsize=11)
ax2.legend(fontsize=8)

# Panel 3: Feature variance distribution (first 3 classification datasets)
ax3 = fig.add_subplot(gs[1, 0])
for name, short, colour in zip(
    ['madelon', 'breast_cancer', 'sonar'],
    ['Madelon', 'BreastCancer', 'Sonar'],
    ['#2196F3', '#4CAF50', '#FF9800']
):
    ds = datasets[name]
    variances = np.var(StandardScaler().fit_transform(ds.X), axis=0)
    ax3.hist(variances, bins=30, alpha=0.6, label=short, color=colour, density=True)
ax3.set_xlabel('Feature variance (standardised)', fontsize=10)
ax3.set_ylabel('Density', fontsize=10)
ax3.set_title('Feature Variance Distributions', fontsize=11)
ax3.legend(fontsize=9)

# Panel 4: Mean absolute feature correlation by dataset
ax4 = fig.add_subplot(gs[1, 1])
mean_corrs = [float(profile_df.loc[name, 'mean_abs_corr']) for name in dataset_names]
ax4.bar(range(len(dataset_names)), mean_corrs, color=colours, alpha=0.85)
ax4.set_xticks(range(len(dataset_names)))
ax4.set_xticklabels(short_names, fontsize=9, rotation=20)
ax4.set_ylabel('Mean |pairwise correlation|', fontsize=10)
ax4.set_title('Feature Correlation Level', fontsize=11)
ax4.axhline(0.3, color='orange', linestyle='--', linewidth=1.2, label='High corr. (>0.3)')
ax4.legend(fontsize=8)

fig.suptitle('Benchmark Dataset Profiles: Key Dimensions', fontsize=14, y=1.01)
plt.savefig('dataset_profiles.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 3: Baseline Models — All Features

Before we do any feature selection, we establish **baseline model performance** using all available features. This is the benchmark we must beat (or at least match) with our selection methods.

We use:
- **Random Forest** — tree-based ensemble, handles mixed features, provides built-in importances
- **LightGBM** — gradient boosting, fast, strong real-world performance, also provides importances

Both are evaluated with 5-fold stratified cross-validation. We report accuracy (classification) and $R^2$ (regression).

In [ ]:
def build_rf_pipeline(task: str, n_estimators: int = 100) -> Pipeline:
    """Build a Random Forest pipeline with standardisation.
    
    Parameters
    ----------
    task : str — 'classification' or 'regression'
    n_estimators : int
    
    Returns
    -------
    sklearn Pipeline
    """
    if task == 'classification':
        model = RandomForestClassifier(
            n_estimators=n_estimators, random_state=42, n_jobs=-1
        )
    else:
        model = RandomForestRegressor(
            n_estimators=n_estimators, random_state=42, n_jobs=-1
        )
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model', model),
    ])


def build_lgbm_pipeline(task: str) -> Pipeline:
    """Build a LightGBM pipeline with standardisation.
    
    Returns None if LightGBM is not installed.
    """
    if not LGBM_AVAILABLE:
        return None
    if task == 'classification':
        model = lgb.LGBMClassifier(
            n_estimators=100, learning_rate=0.1,
            random_state=42, n_jobs=-1, verbose=-1
        )
    else:
        model = lgb.LGBMRegressor(
            n_estimators=100, learning_rate=0.1,
            random_state=42, n_jobs=-1, verbose=-1
        )
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model', model),
    ])


def evaluate_baseline(
    ds: BenchmarkDataset,
    pipeline: Pipeline,
    cv_folds: int = 5
) -> dict:
    """Run cross-validated evaluation on the full feature set.
    
    Parameters
    ----------
    ds : BenchmarkDataset
    pipeline : sklearn Pipeline — RF or LGBM
    cv_folds : int
    
    Returns
    -------
    dict with 'mean_score', 'std_score', 'elapsed_sec', 'metric'
    """
    if ds.task == 'classification':
        scoring = 'accuracy'
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    else:
        scoring = 'r2'
        cv = KFold(n_splits=cv_folds, shuffle=True, random_state=42)

    t0 = time.perf_counter()
    scores = cross_val_score(pipeline, ds.X, ds.y, cv=cv,
                              scoring=scoring, n_jobs=-1)
    elapsed = time.perf_counter() - t0

    return {
        'mean_score': scores.mean(),
        'std_score': scores.std(),
        'elapsed_sec': elapsed,
        'metric': scoring,
    }


print('Baseline evaluation functions defined.')

In [ ]:
# Run baselines for all datasets
# This cell takes 2-4 minutes depending on machine speed
print('Running baseline models on all datasets (5-fold CV)...')
print('This takes approximately 2-4 minutes.')
print('=' * 65)

baseline_results = {}  # nested dict: dataset -> model -> results

for ds_name, ds in datasets.items():
    print(f'\n  {ds.name} (p={ds.p}, n={ds.n})')
    baseline_results[ds_name] = {}

    # Random Forest baseline
    rf_pipe = build_rf_pipeline(ds.task, n_estimators=100)
    print('    Random Forest...', end=' ', flush=True)
    rf_result = evaluate_baseline(ds, rf_pipe)
    baseline_results[ds_name]['Random Forest'] = rf_result
    print(f'{rf_result["mean_score"]:.4f} ± {rf_result["std_score"]:.4f} '
          f'({rf_result["metric"]}) in {rf_result["elapsed_sec"]:.1f}s')

    # LightGBM baseline (if available)
    lgbm_pipe = build_lgbm_pipeline(ds.task)
    if lgbm_pipe is not None:
        print('    LightGBM...', end=' ', flush=True)
        lgbm_result = evaluate_baseline(ds, lgbm_pipe)
        baseline_results[ds_name]['LightGBM'] = lgbm_result
        print(f'{lgbm_result["mean_score"]:.4f} ± {lgbm_result["std_score"]:.4f} '
              f'({lgbm_result["metric"]}) in {lgbm_result["elapsed_sec"]:.1f}s')

print('\nAll baselines complete.')

In [ ]:
# Build a summary table — this is the reference table for all later modules
summary_rows = []
for ds_name, model_results in baseline_results.items():
    ds = datasets[ds_name]
    for model_name, result in model_results.items():
        summary_rows.append({
            'Dataset': ds.name,
            'n': ds.n,
            'p': ds.p,
            'p/n': f'{ds.p_over_n:.3f}',
            'Task': ds.task,
            'Model': model_name,
            'Score': f'{result["mean_score"]:.4f}',
            'Std': f'{result["std_score"]:.4f}',
            'Metric': result['metric'],
            'CV time (s)': f'{result["elapsed_sec"]:.1f}',
        })

summary_df = pd.DataFrame(summary_rows)
print('Baseline Summary Table (All Features, 5-fold CV):')
print('=' * 90)
print(summary_df.to_string(index=False))

# Save for use in later modules
summary_df.to_csv('baseline_results.csv', index=False)
print('\nSaved to baseline_results.csv for later modules.')

In [ ]:
# Visualise baseline performance across datasets and models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

clf_datasets = [name for name, ds in datasets.items() if ds.task == 'classification']
reg_datasets = [name for name, ds in datasets.items() if ds.task == 'regression']

model_colours = {'Random Forest': '#FF9800', 'LightGBM': '#2196F3'}
model_offsets = {'Random Forest': -0.15, 'LightGBM': 0.15}

# Panel 1: Classification datasets
ax = axes[0]
x_pos = np.arange(len(clf_datasets))

for model_name, colour in model_colours.items():
    if not LGBM_AVAILABLE and model_name == 'LightGBM':
        continue
    means = []
    stds = []
    for ds_name in clf_datasets:
        if model_name in baseline_results.get(ds_name, {}):
            r = baseline_results[ds_name][model_name]
            means.append(r['mean_score'])
            stds.append(r['std_score'])
        else:
            means.append(0)
            stds.append(0)
    ax.bar(
        x_pos + model_offsets[model_name], means,
        width=0.28, color=colour, alpha=0.85, label=model_name,
        yerr=stds, capsize=4
    )

clf_labels = [datasets[name].name.replace(' ', '\n') for name in clf_datasets]
ax.set_xticks(x_pos)
ax.set_xticklabels(clf_labels, fontsize=9)
ax.set_ylabel('5-fold CV Accuracy (all features)', fontsize=10)
ax.set_title('Classification Baselines\n(All features, no selection)', fontsize=11)
ax.set_ylim(0.5, 1.02)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0.5, color='grey', linestyle=':', linewidth=1, label='Random (50%)')

# Panel 2: Regression dataset(s)
ax = axes[1]
x_pos_r = np.arange(len(reg_datasets))

for model_name, colour in model_colours.items():
    if not LGBM_AVAILABLE and model_name == 'LightGBM':
        continue
    means = []
    stds = []
    for ds_name in reg_datasets:
        if model_name in baseline_results.get(ds_name, {}):
            r = baseline_results[ds_name][model_name]
            means.append(r['mean_score'])
            stds.append(r['std_score'])
        else:
            means.append(0)
            stds.append(0)
    ax.bar(
        x_pos_r + model_offsets[model_name], means,
        width=0.28, color=colour, alpha=0.85, label=model_name,
        yerr=stds, capsize=4
    )

reg_labels = [datasets[name].name.replace(' ', '\n') for name in reg_datasets]
ax.set_xticks(x_pos_r)
ax.set_xticklabels(reg_labels, fontsize=9)
ax.set_ylabel('5-fold CV R² (all features)', fontsize=10)
ax.set_title('Regression Baselines\n(All features, no selection)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='grey', linestyle=':', linewidth=1, label='Null model (R²=0)')

fig.suptitle(
    'Baseline Model Performance — 5-Fold CV, All Features\n'
    '(These will be compared against selected-feature results in later modules)',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig('baseline_performance.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 4: Dataset-Specific Selection Challenges

Each dataset presents a distinct feature selection challenge. The following analysis highlights what each one will teach us.

In [ ]:
# Analyse the variance structure of each dataset
# Zero-variance or near-zero-variance features are trivially removable
# but their prevalence reveals dataset structure

fig, axes = plt.subplots(1, len(datasets), figsize=(16, 4), sharey=False)

for ax, (name, ds), colour in zip(axes, datasets.items(), colours):
    # Standardise first so variances are on a common scale
    X_std = StandardScaler().fit_transform(ds.X)
    variances = np.sort(np.var(X_std, axis=0))

    ax.bar(
        range(len(variances)), variances,
        color=colour, alpha=0.7, width=1.0
    )
    ax.axhline(0.01, color='red', linestyle='--', linewidth=1,
               label='Near-zero (0.01)')
    ax.set_title(f'{ds.name}\np={ds.p}, n={ds.n}', fontsize=9)
    ax.set_xlabel('Feature (sorted by variance)', fontsize=8)
    if ax == axes[0]:
        ax.set_ylabel('Variance (standardised)', fontsize=9)
    ax.legend(fontsize=7)

fig.suptitle(
    'Feature Variance Profiles — Sorted Ascending\n'
    'Red line: near-zero variance threshold (trivially removable)',
    fontsize=12, y=1.04
)
plt.tight_layout()
plt.savefig('variance_profiles.png', dpi=120, bbox_inches='tight')
plt.show()

# Print zero-variance feature counts
print('\nZero or near-zero variance features (variance < 0.01 after standardisation):')
for name, ds in datasets.items():
    X_std = StandardScaler().fit_transform(ds.X)
    n_low = (np.var(X_std, axis=0) < 0.01).sum()
    print(f'  {ds.name:30s}: {n_low}/{ds.p} features ({n_low/ds.p:.1%})')

In [ ]:
# Print the challenge summary for each dataset — study guide for the course
challenges = {
    'madelon': {
        'selection_challenge': 'Find the 20 informative features in 500 (96% are noise)',
        'key_methods': 'Mutual information, genetic algorithms, wrapper with fast model',
        'main_risk': 'Overfitting the selection process; univariate filters struggle with interactions',
        'modules_featured': '01 (MI filter), 05 (GA), 03 (RFE)',
    },
    'breast_cancer': {
        'selection_challenge': 'Handle highly correlated features (mean/std/worst of same 10 measurements)',
        'key_methods': 'Correlation-aware filters (mRMR), Lasso, stability selection',
        'main_risk': 'Correlated features cause wrapper instability; filter picks all of a correlated group',
        'modules_featured': '01 (correlation filter), 04 (Lasso), 10 (stability)',
    },
    'sonar': {
        'selection_challenge': 'p/n=0.29 — wrapper overfitting is a real danger with only 208 samples',
        'key_methods': 'Regularised embedded methods, stability selection, nested CV',
        'main_risk': 'Selection bias with small n; must use nested CV to avoid optimistic estimates',
        'modules_featured': '03 (wrapper with nested CV), 04 (Elastic Net), 10 (stability)',
    },
    'mnist': {
        'selection_challenge': 'Image pixels — many zero/near-zero features at edges; spatial correlation',
        'key_methods': 'Variance filter, L1 logistic, image-aware methods',
        'main_risk': 'Generic filters ignore spatial structure; many redundant features from adjacent pixels',
        'modules_featured': '01 (variance filter), 08 (high-dimensional), 04 (L1)',
    },
    'communities': {
        'selection_challenge': 'Regression target, missing values, mixed relevance of 122 features',
        'key_methods': 'Imputation first, then F-regression filter, Lasso, RF importance',
        'main_risk': 'Imputation changes feature distributions; selection must happen after imputation',
        'modules_featured': '01 (F-regression filter), 04 (Lasso regression), 11 (production pipelines)',
    },
}

print('\nDataset-Specific Feature Selection Challenges:')
print('=' * 70)
for ds_name, info in challenges.items():
    ds = datasets[ds_name]
    print(f'\n{ds.name} (p={ds.p}, n={ds.n}, p/n={ds.p_over_n:.3f})')
    for key, value in info.items():
        label = key.replace('_', ' ').title()
        print(f'  {label}: {value}')

## Section 5: Convenience Import for Later Modules

The following cell shows how later notebooks in the course will import these datasets. You can copy this pattern into any notebook.

In [ ]:
# Convenience function — single call to load any benchmark dataset
# This is the import pattern used in all later modules

LOADER_REGISTRY = {
    'madelon': load_madelon,
    'breast_cancer': load_breast_cancer_wisconsin,
    'sonar': load_sonar,
    'mnist': load_mnist_subset,
    'communities': load_communities_crime,
}


def load_benchmark(name: str) -> BenchmarkDataset:
    """Load a benchmark dataset by name.
    
    Parameters
    ----------
    name : str — one of: 'madelon', 'breast_cancer', 'sonar', 'mnist', 'communities'
    
    Returns
    -------
    BenchmarkDataset
    
    Example
    -------
    >>> ds = load_benchmark('madelon')
    >>> print(ds.n, ds.p)  # 2600, 500
    """
    if name not in LOADER_REGISTRY:
        raise ValueError(
            f"Unknown dataset '{name}'. "
            f"Available: {list(LOADER_REGISTRY.keys())}"
        )
    return LOADER_REGISTRY[name]()


# Verify the registry works
print('Testing load_benchmark convenience function...')
test_ds = load_benchmark('breast_cancer')
print(f'  Loaded: {test_ds.name}, n={test_ds.n}, p={test_ds.p}')
assert test_ds.n == 569, 'Expected 569 samples'
assert test_ds.p == 30, 'Expected 30 features'
print('  Test passed.')

print('\nIn later modules, import these datasets with:')
print('  from notebooks.02_benchmark_datasets import load_benchmark, BenchmarkDataset')
print('  ds = load_benchmark("madelon")')
print('  X, y = ds.X, ds.y')

## Summary

### Key Takeaways

1. **Five datasets, five different challenges:** Madelon (known noise), Breast Cancer (correlated features), Sonar (small n), MNIST (image pixels with spatial structure), Communities & Crime (missing values, regression). Together they cover the full range of feature selection problems you will encounter in practice.

2. **Baselines are essential:** We have now established the all-features performance for both RF and LightGBM on each dataset. Feature selection methods in later modules will be judged against these baselines. A method that selects 20% of features and matches baseline accuracy is a success.

3. **p/n ratio predicts difficulty:** Sonar (p/n=0.29) will require more care than Breast Cancer (p/n=0.05). The risk table from the complexity guide applies directly.

4. **The `load_benchmark` function is your data layer** for the rest of the course. Every later module uses this exact interface.

### What's Next

- **Module 01:** Statistical filter methods — mutual information, F-statistic, ReliefF, mRMR applied to these datasets with rigorous evaluation.
- **Module 03:** Wrapper methods — RFE, forward selection with proper nested CV to avoid the selection bias pitfall (important for Sonar).

### Going Further

- Load your own dataset using the `BenchmarkDataset` class and add it to the registry.
- Compute the $p/n$ ratio for your dataset and look it up in the risk table from Guide 03.
- Try LightGBM with `num_leaves=15` (smaller trees) on Sonar — does regularising the model improve baseline accuracy on this small dataset?